# 1b. Tensors, with pictures

This is the illustrated companion to [tensor-ladder.md](tensor-ladder.md), and the notebook
[01_the_cache.ipynb](01_the_cache.ipynb) §1.3 pointed here. Same ideas, now with pictures
instead of only tables: what a 1D, 2D, 3D and 4D tensor looks like, and why the *same
shape* can mean two different things depending on how it was built.

Every picture is drawn from real data, small enough to show completely: a few movie-review
sentences turned into vectors by a text encoder, a photo of a cat, and satellite tiles from
the EuroSAT dataset that notebook 01's cache was built from.

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from PIL import Image
from vectormesh.data.vectorizers import Vectorizer

from plotting import plot_stacking

logger.remove()
logger.add(sys.stderr, level="WARNING")

ASSETS = Path("../assets")

## 1D: a vector — one item, one representation

To get real vectors we need an *encoder*: a model that turns a piece of text into a fixed
number of floats. We use `granite-embedding-small-english-r2`, the encoder that built the
IMDB cache you will use in lesson 4. It is small (about 95 MB, downloaded once) and runs on
a laptop CPU.

`vectormesh`'s `Vectorizer` returns one `(C, D)` tensor per text: `C` chunks of `D`
numbers. A single sentence fits in one chunk, so `C = 1` and row 0 is the sentence's vector.

In [ ]:
vectorizer = Vectorizer(model_name="ibm-granite/granite-embedding-small-english-r2", col_name="embed")


def embed(text: str) -> torch.Tensor:
    """One text in, one (D) vector out."""
    (chunks,) = vectorizer([text], batchsize=1)["embed"]  # (C, D), and C = 1 for a sentence
    return chunks[0].float().cpu()


sentence = "A warm, funny film with a cast that clearly enjoyed making it."
vector = embed(sentence)
print(f"{sentence!r}\n-> shape {tuple(vector.shape)}, first five numbers {vector[:5].numpy().round(2)}")

fig, ax = plt.subplots(figsize=(12, 1))
# a few positions are far outside [-2, 2]; clipping the colours keeps the rest visible
ax.imshow(vector.unsqueeze(0), cmap="viridis", aspect="auto", vmin=-2, vmax=2)
ax.set_yticks([])
ax.set_xlabel("position in the vector (0 .. 383)")
ax.set_title(f"1D tensor, shape {tuple(vector.shape)}: one sentence")
plt.show()

384 numbers for one sentence, each one a colour. No single number means anything on its
own. What the encoder was trained for is that sentences with similar meaning get similar
rows of numbers, which is what makes the vector useful as input to a model. A row in a
cache (§6) is exactly this kind of object.

## 2D: the same shape, built two ways

Stack several 1D vectors on top of each other and you get a 2D tensor. There are two
common reasons to do that, and both give the same shape:

- **A batch, `(B, D)`.** Five *different* reviews, each vectorised on its own. They sit
  in one tensor because a model processes many items at once, but they have nothing to do
  with each other.
- **A chunked item, `(C, D)`.** *One* long review, cut into five pieces before
  vectorising. Encoders have a maximum input length, so long documents are split and
  each piece gets its own vector. The five rows belong together: they are one review.

The code below builds both, and the construction is where the meaning lives.

In [ ]:
reviews = [
    "A warm, funny film with a cast that clearly enjoyed making it.",
    "Two hours of my life I will not get back.",
    "The plot is thin, but the photography is beautiful.",
    "My kids watched it three times in one weekend.",
    "A brave ending that most studios would have cut.",
]

review_vectors = [embed(review) for review in reviews]  # five separate (D) vectors
batch = torch.stack(review_vectors)                       # (B, D): one row per review
print(f"{len(review_vectors)} vectors of shape {tuple(review_vectors[0].shape)} -> batch {tuple(batch.shape)}")

plot_stacking(review_vectors, reviews, kind="batch", title="A batch: five different reviews, stacked")
plt.show()

In [ ]:
long_review = (
    "The first hour is slow, with long shots of a farmhouse and a family that barely speaks. "
    "Then the storm arrives and the camera starts to move. "
    "Every small gesture from the first hour comes back with weight. "
    "The lead actress carries the second half without raising her voice. "
    "It runs twenty minutes too long, but I have not stopped thinking about it."
)

pieces = re.split(r"(?<=\.)\s+", long_review)        # one review -> five sentences
chunk_vectors = [embed(piece) for piece in pieces]   # five (D) vectors, all from the same review
chunked = torch.stack(chunk_vectors)                 # (C, D): one row per piece
print(f"1 review -> {len(pieces)} pieces -> chunked {tuple(chunked.shape)}")

plot_stacking(chunk_vectors, pieces, kind="chunks", source="One review", title="A chunked item: one review, cut into five pieces")
plt.show()

We cut on sentences here so you can read the pieces. A real chunker cuts on a fixed number
of tokens (the `Vectorizer` uses the encoder's context window, 512 tokens in lesson 4), but
the result has the same form: one document, `C` rows.

Now compare the two results:

In [ ]:
print(f"batch   {tuple(batch.shape)}  {batch.dtype}")
print(f"chunked {tuple(chunked.shape)}  {chunked.dtype}")
print(f"same shape: {batch.shape == chunked.shape}")

review_vector = chunked.mean(dim=0)  # (C, D) -> (D): one vector for the whole long review
nonsense = batch.mean(dim=0)         # (B, D) -> (D): runs fine, but blends five unrelated reviews
print(f"mean over chunks {tuple(review_vector.shape)}, mean over the batch {tuple(nonsense.shape)}")

`torch` sees no difference: same rank, same shape, same dtype. The difference is in how
they were built, and it decides what you are allowed to do next:

- A **batch** goes through the model row by row. Each review gets its own prediction.
- A **chunked item** is first *aggregated*, for example averaged over its rows, back into
  one `(D)` vector, because the review gets one label, not five.

Both `mean(dim=0)` calls above run without complaint. Averaging the chunks gives a vector
for the long review. Averaging the batch gives a vector for no review at all. This is
[tensor-ladder.md](tensor-ladder.md) §4's point: nothing in the tensor records which of
the two you are holding, so you have to track it yourself, in variable names, comments,
and (for a cache) `metadata.json`.

## 3D: a stack of the 2D picture, two ways

Add a third axis and there are, again, two unrelated things it commonly means.

**Reading 1: a colour image.** `(H, W, 3)`: height, width, three colour channels. A
colour photo really is three 2D grids stacked, one per channel: how much red, green and
blue each pixel has, from 0 to 255.

In [ ]:
cat = torch.from_numpy(np.array(Image.open(ASSETS / "scikit-image" / "chelsea.png")))  # (H, W, 3), uint8
print(f"cat: shape {tuple(cat.shape)}, dtype {cat.dtype}, values {int(cat.min())}..{int(cat.max())}")

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
axes[0].imshow(cat)
axes[0].set_title(f"(H, W, 3) = {tuple(cat.shape)}\nall three channels")
for i, name in enumerate(["red", "green", "blue"]):
    only_this_channel = torch.zeros_like(cat)
    only_this_channel[:, :, i] = cat[:, :, i]  # keep channel i, set the other two to 0
    axes[i + 1].imshow(only_this_channel)
    axes[i + 1].set_title(f"cat[:, :, {i}]: the {name} channel\none (H, W) grid")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

Each channel panel is one `(H, W)` slice of the tensor, drawn in its own colour: bright
where the pixel has a lot of that colour. The cat's orange fur is bright in the red
channel and dark in the blue one. Put the three slices back on top of each other and you
get the photo.

**Reading 2: a batch of chunked items.** `(B, C, D)`: the `(C, D)` chunked review from §2,
now with several reviews in one batch. Nothing about this shape is a photo.

There is a catch you can see coming: reviews have different lengths, so they are cut into
a different number of pieces. `torch.stack` needs every tensor to have the same shape, so
the shorter ones are *padded* with rows of zeros until every review has as many rows as
the longest. `DynamicPadding` from `vectormesh` does exactly that.

In [ ]:
from vectormesh.components import DynamicPadding

three_reviews = [
    long_review,
    "Great soundtrack. Shame about the script. I left before the end.",
    "Not for me. My partner loved it.",
]
chunked_reviews = [torch.stack([embed(s) for s in re.split(r"(?<=\.)\s+", r)]) for r in three_reviews]
print("chunked, one tensor per review:", [tuple(c.shape) for c in chunked_reviews])

batch_chunked = DynamicPadding()(chunked_reviews)  # (B, C_max, D): short reviews get zero rows
print(f"padded into one batch: {tuple(batch_chunked.shape)}")

shown = batch_chunked[:, :, :48].numpy()
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("lightgrey")
fig, axes = plt.subplots(1, len(chunked_reviews), figsize=(13, 2.6))
for b, ax in enumerate(axes):
    n_real = chunked_reviews[b].shape[0]
    rows = np.ma.masked_array(shown[b], mask=np.zeros_like(shown[b], dtype=bool))
    rows.mask[n_real:] = True  # draw padding rows in grey
    ax.imshow(rows, cmap=cmap, aspect="auto", vmin=shown.min(), vmax=shown.max())
    for r in range(n_real, shown.shape[1]):
        ax.text(shown.shape[2] / 2, r, "padding (zeros)", ha="center", va="center", fontsize=8)
    ax.set_title(f"review {b}: {n_real} real chunks")
    ax.set_xticks([])
    ax.set_yticks(range(shown.shape[1]))
    ax.set_ylabel("chunk #" if b == 0 else "")
fig.suptitle(f"(B, C, D) = {tuple(batch_chunked.shape)}: a batch of chunked reviews (first 48 of 384 numbers)")
plt.tight_layout()
plt.show()

Each panel is a whole `(C, D)` item, and the grey rows are the padding. They are drawn the
same way as the colour channels above, as one panel per slice, because the drawing
technique for "a stack of 2D grids" does not depend on what the grids mean.

## 4D: a stack of the 3D picture

One more axis, one more "batch of the previous rung": `(B, H, W, 3)` is a batch of `B`
colour images, each one an `(H, W, 3)` tensor like the cat. Here they are six EuroSAT
satellite tiles of 64 × 64 pixels, the kind of image notebook 01's cache was computed from.

In [ ]:
tile_paths = sorted((ASSETS / "eurosat").glob("*.png"))
tiles = [torch.from_numpy(np.array(Image.open(p))) for p in tile_paths]  # six (H, W, 3) images
batch_images = torch.stack(tiles)                                        # (B, H, W, 3)
print(f"{len(tiles)} tiles of shape {tuple(tiles[0].shape)} -> batch {tuple(batch_images.shape)}")

fig, axes = plt.subplots(1, len(tiles), figsize=(13, 2.4))
for b, ax in enumerate(axes):
    ax.imshow(batch_images[b])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"batch_images[{b}]\n{tile_paths[b].stem.replace('_', ' ')}", fontsize=9)
fig.suptitle(f"(B, H, W, 3) = {tuple(batch_images.shape)}: six images, each (H, W, 3)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
size, offset = 1.0, 0.22
for b in range(len(tiles) - 1, -1, -1):  # draw the last tile first so tile 0 ends up in front
    d = b * offset
    ax.imshow(batch_images[b], extent=(d, d + size, d, d + size), zorder=len(tiles) - b)
    ax.add_patch(plt.Rectangle((d, d), size, size, fill=False, edgecolor="black", zorder=len(tiles) - b))
ax.set_xlim(-0.1, (len(tiles) - 1) * offset + size + 0.1)
ax.set_ylim(-0.1, (len(tiles) - 1) * offset + size + 0.1)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"the same batch as a stack: B = {len(tiles)} images of (H, W, 3)")
plt.show()

## Recap: rank, shape, and (at least) two meanings each

| Rank | Shape | First reading | Second reading |
|:---:|---|---|---|
| 1D | `(D)` | one sentence's vector | — |
| 2D | `(B, D)` | five reviews, stacked | `(C, D)`: one review in five pieces |
| 3D | `(H, W, 3)` | the cat photo | `(B, C, D)`: three chunked reviews, padded |
| 4D | `(B, H, W, 3)` | six satellite tiles | — (rare in this course; see below) |

The pattern repeats at every rung: **going up one rank almost always means "a stack of
the previous rung"**, and the shape never tells you which reading is correct. That lives
in how the tensor was built, in your variable names and comments, and (for a cache you did
not build) in `metadata.json`, as [01_the_cache.ipynb](01_the_cache.ipynb) §1.2 said.

One practical note on 4D: this course rarely needs it directly. Raw images arrive as
`(B, H, W, 3)`, but `ImageVectorizer` collapses each `(H, W, 3)` image to one `(D)` vector
before anything is trained, which is the economic argument from notebook 01 seen from the
shape side. You will meet 4D again only if you batch raw images by hand before encoding
them.

## The same shapes in a cache

A cache stores the vectors so you never run the encoder again. The EuroSAT cache from
notebook 01 holds one `(D)` row per satellite tile, computed by an image encoder
(`dinov2-small`) from tiles like the six above. Batching those rows is the §2 batch
again: a `DataLoader` with `Collate(padder=torch.stack)` stacks `B` rows into `(B, D)`.
Notebook 03 builds its training batches this way.

In [ ]:
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.data import Collate, OneHot

cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")
print(f"one cache row: {tuple(cache['embed'][0].shape)}  <- (D), like the sentence vector in section 1")

onehot = OneHot(num_classes=cache.num_classes, label_col="label", target_col="onehot")
collate = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
loader = DataLoader(cache.dataset.map(onehot), batch_size=6, shuffle=True, collate_fn=collate)

X, y = next(iter(loader))
print(f"one batch: X {tuple(X.shape)}  <- (B, D), six tiles stacked")

fig, ax = plt.subplots(figsize=(10, 2))
ax.imshow(X[:, :64], cmap="viridis", aspect="auto")
ax.set_ylabel("item #")
ax.set_xlabel("first 64 of 384 numbers")
ax.set_title(f"(B, D) = {tuple(X.shape)}: six EuroSAT tiles as cached vectors, batched")
plt.show()

## Exercises

**E1 — three readings, one shape.** You are handed a tensor of shape `(32, 500, 3)` and
nothing else. List at least two plausible readings (e.g. a batch of small RGB thumbnails;
a batch of 3-sensor timeseries, 500 steps long; a batch of documents chunked into 500
pieces with a 3-number summary per chunk). For each, name one extra piece of information,
other than the shape, that would let you tell it apart from the others.

**E2 — why padding exists, and what it costs.** Run `torch.stack(chunked_reviews)` and
read the error. Explain in one sentence why it fails. Then compare
`batch_chunked[2].mean(dim=0)` with `chunked_reviews[2].mean(dim=0)`: they are the
same review, so why do the two vectors differ? What would an aggregator need to know to
average only the real chunks?

**E3 — batch or chunks?** Write a sixth review of three sentences and add it to the §2
batch in two ways: once as a single row (`embed` the whole review), and once as three rows
(`embed` each sentence). `torch` accepts both tensors. Which one still means "one row per
review", and what goes wrong in training with the other?

**E4 — a different real cache.** Repeat the cache cell above for
`pttrn-io/dtd-dinov2-small` instead of EuroSAT. Same shapes, different rows: confirm that
for yourself rather than taking it on faith.